# Part II · Limitation L3 — Binary Exposure Loses Neighbourhood-Intensity Information
**Task 10** | *Worksheet §Part II, Limitation L3 (35 min)*

## The problem

The original exposure indicator
$$E_k(t) = \mathbf{1}[m(k,t) > 0]$$
collapses the entire neighbourhood into a single bit.
A cell with **1 jumping neighbour out of 8** and a cell with **8 jumping neighbours out of 8**
both receive $E_k(t) = 1$.

This discards two kinds of information:
1. **Fractional exposure** $\tilde{E}_k(t) = m(k,t)/n(k,t)$ — how large a fraction of the neighbourhood is jumping.
2. **Ambient signal level** $\hat{E}_k(t) = \bar{S}(k,t)$ — the mean ERK activity of the neighbourhood.

## What we test here

We ask: does the **future self-jump rate increase monotonically** as either of these exposure
quantities grows? A dose–response curve would provide much richer evidence for spatial coordination
than a single $\mathrm{RR}$ value.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'

NODES_PATH = OUTPUT_ROOT / 'exp_1_site_1_ERKKTR_ratio' / 'nodes.csv.gz'
nodes = pd.read_csv(NODES_PATH)
print(f'Loaded {len(nodes):,} nodes')
nodes[['neighbor_count','neighbor_jump_count','neighbor_mean_signal','future_self_jump']].describe()

---
## Toy example — why binary exposure hides the dose

Look at two nodes with very different neighbourhood compositions:

In [ ]:
# Nodes with exactly 1 jumping neighbour (low dose)
low_dose  = nodes[(nodes['neighbor_jump_count'] == 1) & (nodes['neighbor_count'] >= 4)]
# Nodes where ALL neighbours are jumping (high dose)
high_dose = nodes[(nodes['neighbor_jump_count'] == nodes['neighbor_count']) &
                  (nodes['neighbor_count'] >= 4)]

p_low  = low_dose['future_self_jump'].mean()
p_high = high_dose['future_self_jump'].mean()

print(f"n nodes (1 jumping neighbour / ≥4)  : {len(low_dose):,}  →  future jump rate = {p_low:.4f}")
print(f"n nodes (all neighbours jumping / ≥4): {len(high_dose):,}  →  future jump rate = {p_high:.4f}")
print(f"Both labelled E_k(t)=1 by the original estimator.")
print(f"Ratio p_high / p_low = {p_high/p_low:.3f}")

---
## Improvement A — Fractional exposure $\tilde{E}_k(t)$

$$\tilde{E}_k(t) = \frac{m(k,t)}{n(k,t)} \in [0, 1]$$

We bin $\tilde{E}$ into five levels and compute the future-jump rate per bin.

In [ ]:
# Compute fractional exposure (avoid division by zero for isolated cells)
nodes['frac_exposure'] = np.where(
    nodes['neighbor_count'] > 0,
    nodes['neighbor_jump_count'] / nodes['neighbor_count'],
    0.0
)

bins   = [-0.001, 0.0, 0.25, 0.50, 0.75, 1.001]
labels = ['0', '(0, 0.25]', '(0.25, 0.5]', '(0.5, 0.75]', '(0.75, 1]']
nodes['frac_bin'] = pd.cut(nodes['frac_exposure'], bins=bins, labels=labels)

dose_A = (
    nodes.groupby('frac_bin', observed=True)['future_self_jump']
         .agg(future_jump_rate='mean', n_nodes='count')
         .reset_index()
)
dose_A

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: dose-response
axes[0].bar(dose_A['frac_bin'].astype(str), dose_A['future_jump_rate'],
            color='steelblue', alpha=0.85)
axes[0].axhline(nodes['future_self_jump'].mean(), color='grey', ls='--',
                lw=0.9, label='Overall mean')
axes[0].set_xlabel('Fractional exposure $\\tilde{E}_k(t)$')
axes[0].set_ylabel('Future jump rate')
axes[0].set_title('Dose–response: fractional exposure')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=20)

# Right: node counts per bin
axes[1].bar(dose_A['frac_bin'].astype(str), dose_A['n_nodes'],
            color='darkorange', alpha=0.85)
axes[1].set_xlabel('Fractional exposure bin')
axes[1].set_ylabel('Number of nodes')
axes[1].set_title('Nodes per bin')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Graded relative risk: highest bin vs. zero-exposure bin
p0 = dose_A.loc[dose_A['frac_bin'] == '0',          'future_jump_rate'].values[0]
pB = dose_A.loc[dose_A['frac_bin'] == '(0.75, 1]',  'future_jump_rate'].values[0]

RR_tilde = pB / p0 if p0 > 0 else float('nan')
print(f"p_0  (zero exposure)          = {p0:.4f}")
print(f"p_B  (all neighbours jumping) = {pB:.4f}")
print(f"Graded RR̃ = p_B / p_0        = {RR_tilde:.4f}")

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# (a) Is the dose-response curve monotone?
# (b) How does RR̃ compare to the original binary RR from Block 1?
#     Which represents a 'cleaner' signal and why?


---
## Improvement B — Neighbourhood-signal exposure $\hat{E}_k(t) = \bar{S}(k,t)$

Instead of counting jumping neighbours, use the **mean ERK signal** of the neighbourhood
as a continuous exposure variable, discretised into quartiles.

In [ ]:
has_nb = nodes['neighbor_count'] > 0

nodes.loc[has_nb, 'nb_signal_quartile'] = pd.qcut(
    nodes.loc[has_nb, 'neighbor_mean_signal'],
    q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
)

dose_B = (
    nodes.groupby('nb_signal_quartile', observed=True)['future_self_jump']
         .agg(future_jump_rate='mean', n_nodes='count')
         .reset_index()
)
dose_B

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(dose_B['nb_signal_quartile'].astype(str), dose_B['future_jump_rate'],
       color='darkorange', alpha=0.85)
ax.axhline(nodes['future_self_jump'].mean(), color='grey', ls='--',
           lw=0.9, label='Overall mean')
ax.set_xlabel('Neighbourhood mean signal quartile')
ax.set_ylabel('Future jump rate')
ax.set_title('Dose–response: neighbourhood signal level')
ax.legend()
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Does higher ambient ERK signal in the neighbourhood predict more future self-jumping?


### Task 10(c) — Does neighbourhood signal predict jumping *even without* a jumping neighbour?

Filter to unexposed nodes only ($E_k(t)=0$) and repeat the quartile analysis.

In [ ]:
unexposed = nodes[(nodes['neighbor_jump_now'] == False) & has_nb].copy()
unexposed['nb_signal_quartile_unexp'] = pd.qcut(
    unexposed['neighbor_mean_signal'], q=4,
    labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
)

dose_B_unexp = (
    unexposed.groupby('nb_signal_quartile_unexp', observed=True)['future_self_jump']
             .agg(future_jump_rate='mean', n_nodes='count')
             .reset_index()
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(dose_B_unexp['nb_signal_quartile_unexp'].astype(str),
       dose_B_unexp['future_jump_rate'],
       color='mediumseagreen', alpha=0.85)
ax.set_xlabel('Neighbourhood signal quartile (unexposed nodes only)')
ax.set_ylabel('Future jump rate')
ax.set_title('Signal-level effect among unexposed cells')
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Is there still a dose-response among unexposed cells?
# What does this suggest about the role of ambient ERK signal
# independent of discrete jump events?


---
## ★ Bonus — Combine L1 and L3: lagged fractional exposure

Define the **lagged fractional exposure**:
$$\tilde{E}_k^{(\tau)}(t) = \frac{m(k,\, t-\tau)}{n(k,\, t-\tau)}$$
and the **graded lagged relative risk**:
$$\widetilde{\mathrm{RR}}^{(\tau)} = \frac{p_B^{(\tau)}}{p_0^{(\tau)}}$$
where $p_B^{(\tau)}$ is the future-jump rate in the highest fractional-exposure bin at lag $\tau$,
and $p_0^{(\tau)}$ is the rate in the zero-exposure bin.

In [ ]:
nodes = nodes.sort_values(['track_id', 'Image_Metadata_T']).reset_index(drop=True)

MAX_LAG = 5
rr_tilde_lags = []

for tau in range(MAX_LAG + 1):
    if tau == 0:
        frac_col = 'frac_exposure'  # already computed above
    else:
        # ── YOUR CODE ────────────────────────────────────────────────────────
        # Construct the lagged fractional exposure:
        # shift 'frac_exposure' forward by tau within each track.
        frac_col = f'frac_exp_lag{tau}'
        nodes[frac_col] = (
            nodes.groupby('track_id')['frac_exposure']
                 .shift(tau)
                 .fillna(0.0)
        )

    bins   = [-0.001, 0.0, 0.25, 0.50, 0.75, 1.001]
    labels_b = ['0', '(0,0.25]', '(0.25,0.5]', '(0.5,0.75]', '(0.75,1]']
    tmp_bin = pd.cut(nodes[frac_col], bins=bins, labels=labels_b)

    rates = nodes.groupby(tmp_bin, observed=True)['future_self_jump'].mean()
    p0_t = rates.get('0', np.nan)
    pB_t = rates.get('(0.75,1]', np.nan)
    rr_t = pB_t / p0_t if p0_t > 0 else np.nan
    rr_tilde_lags.append({'τ (frames)': tau, 'τ (min)': tau*5,
                           'p_0': round(p0_t,4), 'p_B': round(pB_t,4),
                           'RR_tilde': round(rr_t,4)})

df_tilde = pd.DataFrame(rr_tilde_lags).set_index('τ (frames)')
print(df_tilde)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(df_tilde['τ (min)'], df_tilde['RR_tilde'], 'o-', color='purple', lw=2,
        label='$\\widetilde{\\mathrm{RR}}^{(\\tau)}$ (graded + lagged)')
ax.axhline(1, color='grey', ls='--', lw=0.8)
ax.set_xlabel('Lag $\\tau$ (min)')
ax.set_ylabel('Graded lagged RR')
ax.set_title('Combined L1+L3 estimator')
ax.legend()
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Does the combined estimator peak at the same τ* as the binary lagged RR from L1?
# Is it larger or smaller than the binary version at the same lag?
